# Explainable Credit Risk Prediction
Independent portfolio project using UCI German Credit data (CC BY 4.0). This is not the graded Lending Club assignment. No model scores are claimed before execution.

## 1. Load the public dataset
Bad credit-risk label = 1; good credit-risk label = 0.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import matplotlib.pyplot as plt
data = fetch_ucirepo(id=144)
X = data.data.features.copy()
y = (data.data.targets.squeeze().astype(int) == 2).astype(int)
print("Rows and features:", X.shape)
print("Class distribution:", y.value_counts().to_dict())
display(X.head())

## 2. EDA and quality checks
Examine feature types, missingness, and label balance.

In [ ]:
display(pd.DataFrame({"dtype": X.dtypes.astype(str), "missing": X.isna().sum()}))
y.value_counts(normalize=True).sort_index().plot.bar(title="Credit risk labels: 0=good, 1=bad")
plt.ylabel("Fraction")
plt.show()

## 3. Train, tune, calibrate, and evaluate
The separate `src/train.py` pipeline compares regular and class-weighted Logistic Regression and Random Forest, uses stratified five-fold CV within the training set, fits sigmoid calibration from training folds, chooses an F1 threshold using out-of-fold predictions, and evaluates only on the held-out test set. SHAP explanations are optional.

In [ ]:
import sys
from pathlib import Path
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))
from src.train import run
report = run(output_dir=str(root / "results"), shap_enabled=True)

## 4. Inspect held-out results
ROC-AUC and average precision are ranking metrics; Brier/log loss assess probabilities. F1, precision, and recall use a threshold selected on training-only OOF predictions. Do not tune the method on the test results.

In [ ]:
from IPython.display import Image, display
display(pd.DataFrame([report["test"]]).T.rename(columns={0: "Independent test"}))
display(Image(filename=str(root / "results" / "evaluation.png")))

## 5. Limitations
The data are a historical benchmark, not prospective loan defaults. Calibration and SHAP results may be unstable with few observations. Fairness, external validation, economic costs, and regulatory requirements are outside scope. SHAP describes the underlying uncalibrated model and does not establish causality.